In [ ]:
import os
import re
import pandas as pd
import numpy as np
import json
import warnings
from multiprocessing import cpu_count
from tqdm.auto import tqdm
from collections import Counter
from scipy import stats
import matplotlib.pyplot as plt
import math

In [ ]:
"""
Boilerplate cleaning

Usage:
- df_aj = pd.read_csv("aljazeera.csv")
- df_cnn = pd.read_csv("cnn.csv")
- df_aj = remove_boilerplate_df(df_aj, outlet="aljazeera")
- df_cnn = remove_boilerplate_df(df_cnn, outlet="cnn")
- save back to CSV/parquet
"""


# -----------------------------
# Helper: normalize whitespace
# -----------------------------
def _normalize_ws(s: str) -> str:
    s = re.sub(r"\r\n?", "\n", s)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


# -----------------------------
# Boilerplate patterns
# -----------------------------
# Notes:
# - We prefer conservative patterns (lower risk of deleting real content)
# - We mostly target sections starting with a known boilerplate cue
#   and remove to end, or remove single lines.
#
# You can add patterns as you discover repeated noise.

# Remove blocks "from cue to end of document"
CUT_TO_END_PATTERNS_COMMON = [
    r"(?is)\n{0,2}\s*subscribe to .*?$",  # "Subscribe to..."
    r"(?is)\n{0,2}\s*sign up for .*?$",
    r"(?is)\n{0,2}\s*download the app.*?$",
    r"(?is)\n{0,2}\s*cookie(s)? settings.*?$",
    r"(?is)\n{0,2}\s*privacy policy.*?$",
    r"(?is)\n{0,2}\s*terms of (use|service).*?$",
]

# Remove single-line / short recurring prompts
LINE_PATTERNS_COMMON = [
    r"(?im)^\s*follow us\s*$",
    r"(?im)^\s*follow\s+.*(facebook|x|twitter|instagram|tiktok|youtube).*$",
    r"(?im)^\s*share\s+this\s*$",
    r"(?im)^\s*read more\s*$",
    r"(?im)^\s*advertisement\s*$",
    r"(?im)^\s*sponsored\s*$",
    r"(?im)^\s*related topics\s*:.*$",
]

# Outlet-specific: Al Jazeera recurring phrases often included in scraped text
CUT_TO_END_PATTERNS_AJ = [
    r"(?is)\n{0,2}\s*follow al jazeera english.*?$",
    r"(?is)\n{0,2}\s*follow al jazeera.*?$",
    r"(?is)\n{0,2}\s*al jazeera english.*?$",
]

LINE_PATTERNS_AJ = [
    r"(?im)^\s*follow al jazeera english\s*$",
    r"(?im)^\s*follow al jazeera\s*$",
]

# Outlet-specific: CNN sometimes has “Sign up for CNN’s …”, “Read more”, etc.
CUT_TO_END_PATTERNS_CNN = [
    r"(?is)\n{0,2}\s*sign up for cnn.*?$",
    r"(?is)\n{0,2}\s*read more from cnn.*?$",
    r"(?is)\n{0,2}\s*cnn’s\s+.*newsletter.*?$",
]

LINE_PATTERNS_CNN = [
    r"(?im)^\s*cnn.*newsletter.*$",
    r"(?im)^\s*want to read more.*$",
]


def remove_boilerplate_text(text: str, outlet: str | None = None) -> str:
    """
    Remove boilerplate from a single article text.
    outlet: "aljazeera" or "cnn" (optional)
    """
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""

    s = str(text)
    s = _normalize_ws(s)

    # 1) Remove obvious repeated lines
    for pat in LINE_PATTERNS_COMMON:
        s = re.sub(pat, "", s)

    if outlet == "aljazeera":
        for pat in LINE_PATTERNS_AJ:
            s = re.sub(pat, "", s)
    elif outlet == "cnn":
        for pat in LINE_PATTERNS_CNN:
            s = re.sub(pat, "", s)

    s = _normalize_ws(s)

    # 2) Cut to end if we hit known boilerplate cues
    cut_patterns = CUT_TO_END_PATTERNS_COMMON.copy()
    if outlet == "aljazeera":
        cut_patterns += CUT_TO_END_PATTERNS_AJ
    elif outlet == "cnn":
        cut_patterns += CUT_TO_END_PATTERNS_CNN

    for pat in cut_patterns:
        # If a cue appears, cut everything from cue to end
        m = re.search(pat, s)
        if m:
            s = s[: m.start()]
            s = _normalize_ws(s)

    # 3) Final cleanup: strip leftover short junk lines
    # Remove lines that are extremely short and look like navigation
    lines = [ln.strip() for ln in s.split("\n")]
    cleaned_lines = []
    for ln in lines:
        if not ln:
            cleaned_lines.append("")
            continue
        # Drop very short nav-like lines
        if len(ln) <= 3:
            continue
        if re.fullmatch(r"(?i)(home|news|video|live|more|menu)", ln):
            continue
        cleaned_lines.append(ln)

    s = "\n".join(cleaned_lines)
    s = _normalize_ws(s)

    return s


def remove_boilerplate_df(df: pd.DataFrame, outlet: str, text_col: str = "text_clean") -> pd.DataFrame:
    """
    Apply boilerplate removal to a dataframe.
    Creates:
      - text_noboiler
      - text_noboiler_len
    """
    df = df.copy()

    # if text_clean missing, fallback to text
    if text_col not in df.columns:
        text_col = "text" if "text" in df.columns else text_col

    df["text_noboiler"] = df[text_col].apply(lambda t: remove_boilerplate_text(t, outlet=outlet))
    df["text_noboiler_len"] = df["text_noboiler"].astype(str).str.len()

    return df


# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    df_aj = pd.read_csv("aljazeera.csv")
    df_cnn = pd.read_csv("cnn.csv")

    df_aj = remove_boilerplate_df(df_aj, outlet="aljazeera", text_col="text_clean")
    df_cnn = remove_boilerplate_df(df_cnn, outlet="cnn", text_col="text_clean")

    # Optional: overwrite text_clean with cleaned version
    # df_aj["text_clean"] = df_aj["text_noboiler"]
    # df_cnn["text_clean"] = df_cnn["text_noboiler"]

    df_aj.to_csv("aljazeera_cleaned.csv", index=False)
    df_cnn.to_csv("cnn_cleaned.csv", index=False)

    print("Saved cleaned files:",
          "aljazeera_cleaned.csv",
          "cnn_cleaned.csv")

In [ ]:
"""
- Sentiment: spaCy POS-filtered lemmatized text -> VADER
- Word2Vec: NLTK stopwords + WordNet lemmatizer preprocessing
           -> Word2Vec hyperparams (vector_size=200, window=40, sample=0.01,
              epochs=200, negative=10, min_count=10)

INPUT FILES:
- aljazeera_cleaned.csv
- cnn_cleaned.csv

Expected columns in CSV:
- url, title, text, date

OUTPUTS (written into ./outputs_updated/):
- sent_by_outlet.csv
- sent_by_month_outlet.csv                (mean compound by month/outlet)
- sent_label_counts_by_month_outlet.csv   (pos/neg counts by month/outlet)
- w2v_neighbors.json                      (neighbors per outlet)
- w2v_overlap.json                        (Jaccard overlap per word)
- articles_enriched_updated.csv
- articles_enriched_updated.parquet
"""

warnings.filterwarnings("ignore", category=UserWarning)

# =========================
# CONFIG
# =========================
PATH_AJ = "aljazeera_cleaned.csv"
PATH_CNN = "cnn_cleaned.csv"

OUTPUT_DIR = "./outputs_updated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sentiment config
SPACY_MODEL = "en_core_web_sm"
KEEP_POS = {"NOUN", "VERB", "ADJ", "ADV", "PROPN"}

# Word2Vec config (from your notebooks)
W2V_VECTOR_SIZE = 200
W2V_WINDOW = 20
W2V_SAMPLE = 0.01
W2V_EPOCHS = 50
W2V_NEGATIVE = 10
W2V_MIN_COUNT = 3

# ---- Word2Vec cleaning additions ----
W2V_MIN_TOKEN_LEN = 4

# domain / web junk + common boilerplate residues
W2V_DROP_TOKENS = {
    "http","https","www","com","org","net","amp",
    "jpg","jpeg","png","gif","svg","pdf",
    "utm","utm_source","utm_medium","utm_campaign",
}

# keep ONLY ASCII a-z tokens to remove Hebrew/Arabic fragments etc.
W2V_ASCII_ONLY = True

# filter to thesis window (post Oct 7, 2023)
FILTER_POST_OCT7 = True
OCT7_TS = pd.Timestamp("2023-10-07", tz="UTC")
# Which terms to extract neighbors for
TARGET_WORDS = ["gaza", "israel", "hamas", "idf", "palestine", "terrorist", "army", "netanyahu", "minister", "ministry", "defence", "president"]
TOPN_NEIGHBORS = 20
TOPN_OVERLAP = 20  # overlap computed on top-N neighbors


# =========================
# LOAD DATA
# =========================
def safe_read_csv(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, engine="python", on_bad_lines="skip")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    # common weird column name
    if "date;;;" in df.columns:
        df = df.rename(columns={"date;;;": "date"})
    return df


def load_and_merge() -> pd.DataFrame:

    df_aj = normalize_columns(safe_read_csv(PATH_AJ))
    df_cnn = normalize_columns(safe_read_csv(PATH_CNN))

    required = {"url", "title", "text", "date"}
    for name, df in [("AlJazeera", df_aj), ("CNN", df_cnn)]:
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{name} missing columns: {missing}. Found={list(df.columns)}")

    df_aj["outlet"] = "aljazeera"
    df_cnn["outlet"] = "cnn"

    df = pd.concat([df_aj, df_cnn], ignore_index=True)
        
    # parse date (keep UTC)
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
    df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

    # Filter to thesis window (post Oct 7, 2023)
    if FILTER_POST_OCT7:
        df = df[df["date"] >= OCT7_TS].copy()
    # basic hygiene
    df["text"] = df["text"].astype(str)
    df["title"] = df["title"].astype(str)
    df["url"] = df["url"].astype(str)

    return df


# =========================
# SENTIMENT: spaCy POS-lemmas + VADER
# =========================
def setup_sentiment():
    import nltk
    from nltk.sentiment.vader import SentimentIntensityAnalyzer

    nltk.download("vader_lexicon", quiet=True)

    import spacy
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])  # fast

    sid = SentimentIntensityAnalyzer()
    return nlp, sid


def spacy_lemma_pos_clean(nlp, text: str) -> str:
    if not text:
        return ""
    doc = nlp(text)
    lemmas = [t.lemma_ for t in doc if t.pos_ in KEEP_POS]
    return " ".join(lemmas)


def vader_scores(sid, text: str) -> dict:
    if not text:
        return {"neg": 0.0, "neu": 0.0, "pos": 0.0, "compound": 0.0}
    return sid.polarity_scores(text)


def add_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    nlp, sid = setup_sentiment()

    tqdm.pandas(desc="spaCy lemma+POS clean")
    df = df.copy()
    df["clean_text_spacy"] = df["text"].progress_apply(lambda t: spacy_lemma_pos_clean(nlp, t))

    tqdm.pandas(desc="VADER on clean text")
    scores = df["clean_text_spacy"].progress_apply(lambda t: vader_scores(sid, t))

    df["sent_neg_clean"] = scores.apply(lambda s: s["neg"])
    df["sent_neu_clean"] = scores.apply(lambda s: s["neu"])
    df["sent_pos_clean"] = scores.apply(lambda s: s["pos"])
    df["sent_compound_clean"] = scores.apply(lambda s: s["compound"])

    # Your notebook-style binary label (compound > 0)
    df["sent_label_clean"] = np.where(df["sent_compound_clean"] > 0, "pos", "neg")

    return df


def export_sentiment_outputs(df: pd.DataFrame):
    # overall by outlet
    sent_by_outlet = (
        df.groupby("outlet")["sent_compound_clean"]
          .agg(["count", "mean", "median", "std"])
          .reset_index()
          .rename(columns={
              "count": "n_articles",
              "mean": "mean_compound",
              "median": "median_compound",
              "std": "std_compound"
          })
    )
    sent_by_outlet.to_csv(os.path.join(OUTPUT_DIR, "sent_by_outlet.csv"), index=False)

    # mean compound by month + outlet
    sent_by_month_outlet = (
        df.dropna(subset=["month"])
          .groupby(["outlet", "month"])["sent_compound_clean"]
          .mean()
          .reset_index(name="mean_compound")
    )
    sent_by_month_outlet.to_csv(os.path.join(OUTPUT_DIR, "sent_by_month_outlet.csv"), index=False)

    # pos/neg counts by month + outlet
    sent_label_counts = (
        df.dropna(subset=["month"])
          .groupby(["outlet", "month", "sent_label_clean"])
          .size()
          .reset_index(name="n")
    )
    sent_label_counts.to_csv(os.path.join(OUTPUT_DIR, "sent_label_counts_by_month_outlet.csv"), index=False)


# =========================
# WORD2VEC: NLTK preprocessing + gensim
# =========================
def setup_w2v():
    import nltk
    nltk.download("punkt", quiet=True)
    nltk.download("stopwords", quiet=True)
    nltk.download("wordnet", quiet=True)

    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    from nltk.tokenize import word_tokenize

    stop_words = set(stopwords.words("english"))
    lemm = WordNetLemmatizer()

    return stop_words, lemm, word_tokenize

def w2v_preprocess(text, stop_words, lemm, word_tokenize):
    if not text:
        return []

    text = re.sub(r"[^\w\s]", " ", text)
    tokens = word_tokenize(text)

    cleaned = []
    for w in tokens:
        w = w.lower().strip()

        if W2V_ASCII_ONLY:
            try:
                w.encode("ascii")
            except UnicodeEncodeError:
                continue

        if not w.isalpha():
            continue

        if w in stop_words or w in W2V_DROP_TOKENS:
            continue

        # allow some short but meaningful tokens
        if w in {"idf"}:
            cleaned.append(w)
            continue

        if len(w) < W2V_MIN_TOKEN_LEN:
            continue

        w = lemm.lemmatize(w)

        if w in stop_words or w in W2V_DROP_TOKENS:
            continue

        if len(w) < W2V_MIN_TOKEN_LEN:
            continue

        cleaned.append(w)

    return cleaned

def train_word2vec(sentences):
    from gensim.models import Word2Vec
    model = Word2Vec(
        vector_size=W2V_VECTOR_SIZE,
        window=W2V_WINDOW,
        sample=W2V_SAMPLE,
        epochs=W2V_EPOCHS,
        negative=W2V_NEGATIVE,
        min_count=W2V_MIN_COUNT,
        workers=cpu_count(),
        hs=0
    )
    model.build_vocab(sentences)
    model.train(sentences, total_examples=model.corpus_count, epochs=model.epochs)
    return model


def get_neighbors(model, word: str, topn: int = 20):
    if word not in model.wv:
        return []
    return [(w, float(sim)) for (w, sim) in model.wv.most_similar(word, topn=topn)]


def neighbor_set(model, word: str, topn: int = 20):
    return {w for (w, _) in get_neighbors(model, word, topn=topn)}


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def build_w2v_models(df: pd.DataFrame):
    stop_words, lemm, word_tokenize = setup_w2v()

    models = {}
    for outlet in ["aljazeera", "cnn"]:
        sub = df[df["outlet"] == outlet].copy()
        tqdm.pandas(desc=f"W2V preprocess ({outlet})")
        sub["tokens"] = sub["text"].progress_apply(lambda t: w2v_preprocess(t, stop_words, lemm, word_tokenize))

        sentences = sub["tokens"].tolist()
        print(f"[W2V] Training {outlet}: sentences={len(sentences)}")
        models[outlet] = train_word2vec(sentences)

    return models


def export_w2v_outputs(models: dict):
    # neighbors per outlet
    neighbors_out = {}
    for outlet, model in models.items():
        neighbors_out[outlet] = {}
        for w in TARGET_WORDS:
            neighbors_out[outlet][w] = get_neighbors(model, w, topn=TOPN_NEIGHBORS)

    with open(os.path.join(OUTPUT_DIR, "w2v_neighbors.json"), "w", encoding="utf-8") as f:
        json.dump(neighbors_out, f, ensure_ascii=False, indent=2)

    # overlap (robust cross-outlet comparison)
    overlap_out = {}
    aj_model = models["aljazeera"]
    cnn_model = models["cnn"]

    for w in TARGET_WORDS:
        aj_set = neighbor_set(aj_model, w, topn=TOPN_OVERLAP)
        cnn_set = neighbor_set(cnn_model, w, topn=TOPN_OVERLAP)
        overlap_out[w] = {
            "topn": TOPN_OVERLAP,
            "jaccard_overlap": jaccard(aj_set, cnn_set),
            "aljazeera_only": sorted(list(aj_set - cnn_set)),
            "cnn_only": sorted(list(cnn_set - aj_set)),
            "common": sorted(list(aj_set & cnn_set)),
        }

    with open(os.path.join(OUTPUT_DIR, "w2v_overlap.json"), "w", encoding="utf-8") as f:
        json.dump(overlap_out, f, ensure_ascii=False, indent=2)

    # save models too
    models["aljazeera"].save(os.path.join(OUTPUT_DIR, "aljazeera_word2vec.model"))
    models["cnn"].save(os.path.join(OUTPUT_DIR, "cnn_word2vec.model"))


# =========================
# SAVE ENRICHED
# =========================
def export_enriched(df: pd.DataFrame):
    df.to_csv(os.path.join(OUTPUT_DIR, "articles_enriched_updated.csv"), index=False)

    # optional parquet
    try:
        df.to_parquet(os.path.join(OUTPUT_DIR, "articles_enriched_updated.parquet"), index=False)
        print("[OK] Saved parquet: articles_enriched_updated.parquet")
    except Exception as e:
        print("[WARN] Could not save parquet (install pyarrow):", e)


# =========================
# MAIN
# =========================
def main():
    print("[1/4] Loading CSVs and merging...")
    df = load_and_merge()
    # Filter to thesis window (post Oct 7, 2023)
    df = df.dropna(subset=["date"]).copy()
    df = df[df["date"] >= "2023-10-07"].copy()
    print("Rows:", len(df))
    print(df["outlet"].value_counts())

    print("\n[2/4] Computing sentiment (spaCy lemmas + VADER)...")
    df = add_sentiment(df)
    export_sentiment_outputs(df)
    print("[OK] Sentiment outputs written in", OUTPUT_DIR)

    print("\n[3/4] Training Word2Vec models (per outlet)...")
    models = build_w2v_models(df)
    export_w2v_outputs(models)
    print("[OK] Word2Vec outputs written in", OUTPUT_DIR)

    print("\n[4/4] Saving enriched dataset...")
    export_enriched(df)
    print("\nDONE ✅ Outputs in:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

In [ ]:
"""
Outputs:
- sentiment_summary_by_outlet.csv
- sentiment_by_month_outlet.csv
- sentiment_tests.txt
- entity_focus.csv
- entity_tests.csv
- lexical_keyterms_by_month.csv
- log_odds_top_words_aljazeera.csv / log_odds_top_words_cnn.csv
- w2v_neighbors.json + w2v_overlap.json (if models load)
"""

OUTDIR = "./full_data_outputs"
os.makedirs(OUTDIR, exist_ok=True)

PARQUET_PATH = "articles_enriched_updated.parquet"

# -------------------------
# Load
# -------------------------
df = pd.read_parquet(PARQUET_PATH).copy()

# Ensure core cols
for col in ["outlet", "date", "text"]:
    if col not in df.columns:
        raise ValueError(f"Missing column '{col}' in parquet. Found: {list(df.columns)}")

df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df = df.dropna(subset=["date"]).copy()
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

df["text"] = df["text"].astype(str)
df["outlet"] = df["outlet"].astype(str).str.lower()

# Filter to thesis window (post Oct 7, 2023)
df = df[df["date"] >= pd.Timestamp("2023-10-07", tz="UTC")].copy()

print("Counts post-2023-10-07:\n", df["outlet"].value_counts())

# -------------------------
# Tokenization for counts (simple, consistent)
# -------------------------
TOKEN_RE = re.compile(r"[A-Za-z]+")
STOP_MINLEN = 3

def tokenize(text: str):
    toks = TOKEN_RE.findall(text.lower())
    toks = [t for t in toks if len(t) >= STOP_MINLEN]
    return toks

tqdm.pandas(desc="Tokenizing")
df["tokens"] = df["text"].progress_apply(tokenize)
df["n_tokens"] = df["tokens"].apply(len).astype(int)

# -------------------------
# SENTIMENT (distributional)
# -------------------------
if "sent_compound_clean" not in df.columns:
    raise ValueError("Expected 'sent_compound_clean' in parquet. Re-run sentiment pipeline first.")

sent_summary = (
    df.groupby("outlet")["sent_compound_clean"]
      .agg(
          n="count",
          mean="mean",
          median="median",
          q25=lambda x: x.quantile(0.25),
          q75=lambda x: x.quantile(0.75),
          std="std",
      )
      .reset_index()
)
sent_summary.to_csv(os.path.join(OUTDIR, "sentiment_summary_by_outlet.csv"), index=False)

sent_by_month = (
    df.groupby(["outlet", "month"])["sent_compound_clean"]
      .agg(n="count", mean="mean", median="median")
      .reset_index()
)

def bootstrap_ci_mean(x, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    if len(x) < 2:
        return (np.nan, np.nan)
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(x, size=len(x), replace=True)
        boots.append(np.mean(samp))
    lo = np.quantile(boots, alpha / 2)
    hi = np.quantile(boots, 1 - alpha / 2)
    return float(lo), float(hi)

rows = []
for (outlet, month), g in tqdm(df.groupby(["outlet", "month"]), desc="Bootstrapping monthly CI"):
    lo, hi = bootstrap_ci_mean(g["sent_compound_clean"].values)
    rows.append({"outlet": outlet, "month": month, "mean_ci_lo": lo, "mean_ci_hi": hi})

ci_df = pd.DataFrame(rows)
sent_by_month = sent_by_month.merge(ci_df, on=["outlet", "month"], how="left")
sent_by_month.to_csv(os.path.join(OUTDIR, "sentiment_by_month_outlet.csv"), index=False)

def cliffs_delta(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    return (gt - lt) / (len(x) * len(y))

aj = df[df["outlet"] == "aljazeera"]["sent_compound_clean"].values
cnn = df[df["outlet"] == "cnn"]["sent_compound_clean"].values

mw = stats.mannwhitneyu(aj, cnn, alternative="two-sided")
cd = cliffs_delta(aj, cnn)

with open(os.path.join(OUTDIR, "sentiment_tests.txt"), "w", encoding="utf-8") as f:
    f.write("Mann–Whitney U test (sent_compound_clean)\n")
    f.write(f"U={mw.statistic:.3f}, p={mw.pvalue:.6g}\n\n")
    f.write("Cliff's delta (AJ vs CNN)\n")
    f.write(f"delta={cd:.4f}  (positive => AJ more positive than CNN)\n")

# -------------------------
# ENTITY FOCUS (presence + rate per 10k tokens)  [FIXED]
# -------------------------
df["text_lc"] = df["text"].astype(str).str.lower()

# Notes:
# - We use NON-capturing groups (?:...) so findall() returns full matches, not tuples.
# - We include "israel's" / "israels" variants via israel(?:'s|s)?
# - We include both "Israel Defense Forces" and "Israeli Defense Forces"
# - We include "Israeli army/military/forces" and possessive "Israel's army/military"
ENTITY_PATTERNS = {
    "israel": r"\bisrael\b",
    "palestine": r"\bpalestine\b",
    "gaza": r"\bgaza\b",
    "hamas": r"\bhamas\b",
    "terrorist": r"\bterrorist(?:s)?\b", 
    "army": r"\bgaza\b", 
    "netanyahu": r"\bnetanyahu\b", 
    "minister": r"\bminister\b", 
    "ministry": r"\bministry\b", 
    "defence": r"\bgaza\b",
    "president": r"\bpresident\b",
    "idf": (
        r"\b(?:"
        r"idf|i\.?d\.?f\.?"
        r"|(?:israel(?:'s|s)?|israeli)\s+defen[cs]e\s+forces"
        r"|(?:israel(?:'s|s)?|israeli)\s+forces"
        r"|(?:israel(?:'s|s)?|israeli)\s+military"
        r"|(?:israel(?:'s|s)?|israeli)\s+army"
        r")\b"
    ),
}

ENTITY_REGEX = {k: re.compile(v, flags=re.IGNORECASE) for k, v in ENTITY_PATTERNS.items()}
ENTITIES = list(ENTITY_REGEX.keys())

# Presence + mention counts
for ent, rx in ENTITY_REGEX.items():
    df[f"has_{ent}"] = df["text_lc"].apply(lambda s: int(bool(rx.search(s))))
    # finditer is safest for counting regardless of grouping/regex engine details
    df[f"count_{ent}"] = df["text_lc"].apply(lambda s: sum(1 for _ in rx.finditer(s)))

# Presence % by outlet
presence = (
    df.groupby("outlet")[[f"has_{e}" for e in ENTITIES]]
      .mean()
      .reset_index()
      .rename(columns={f"has_{e}": f"pct_articles_mention_{e}" for e in ENTITIES})
)
for e in ENTITIES:
    presence[f"pct_articles_mention_{e}"] *= 100.0

# Mentions per 10k tokens by outlet
rates_rows = []
for outlet, g in df.groupby("outlet"):
    total_tokens = g["n_tokens"].sum()
    row = {"outlet": outlet, "total_tokens": int(total_tokens)}
    for ent in ENTITIES:
        total_mentions = g[f"count_{ent}"].sum()
        row[f"mentions_per_10k_{ent}"] = (total_mentions / total_tokens) * 10000 if total_tokens else np.nan
    rates_rows.append(row)

rates = pd.DataFrame(rates_rows)

entity_focus = presence.merge(rates.drop(columns=["total_tokens"]), on="outlet", how="left")
entity_focus.to_csv(os.path.join(OUTDIR, "entity_focus.csv"), index=False)

# Chi-square tests on presence + Cramer's V
def cramers_v(chi2, n, r, k):
    return math.sqrt((chi2 / n) / (min(r - 1, k - 1)))

tests = []
for ent in ENTITIES:
    tab = pd.crosstab(df["outlet"], df[f"has_{ent}"])
    if tab.shape[1] < 2:
        continue
    chi2, p, dof, exp = stats.chi2_contingency(tab)
    v = cramers_v(chi2, tab.to_numpy().sum(), tab.shape[0], tab.shape[1])
    tests.append({"entity": ent, "chi2": chi2, "p": p, "cramers_v": v})

pd.DataFrame(tests).to_csv(os.path.join(OUTDIR, "entity_tests.csv"), index=False)
print("[OK] wrote entity_focus.csv + entity_tests.csv")

# -------------------------
# LEXICAL FRAMING: key terms per 10k tokens + log-odds
# -------------------------
KEY_TERMS = [
    "militant", "militants", "fighter", "fighters",
    "attack", "attacks", "operation", "operations",
    "crisis", "settler", "settlers", "apartheid", "genocide",
    "terrorist", "terrorists", "ceasefire"
]

def term_count(tokens, term):
    return sum(1 for t in tokens if t == term)

lex_rows = []
for (outlet, month), g in df.groupby(["outlet", "month"]):
    total_tokens = g["n_tokens"].sum()
    row = {"outlet": outlet, "month": month, "n_articles": len(g), "total_tokens": int(total_tokens)}
    for term in KEY_TERMS:
        c = g["tokens"].apply(lambda toks: term_count(toks, term)).sum()
        row[f"{term}_per_10k"] = (c / total_tokens) * 10000 if total_tokens else np.nan
    lex_rows.append(row)

lex_df = pd.DataFrame(lex_rows)
lex_df.to_csv(os.path.join(OUTDIR, "lexical_keyterms_by_month.csv"), index=False)

def log_odds_with_prior(counts_a, counts_b, alpha=0.01):
    vocab = set(counts_a.keys()) | set(counts_b.keys())
    a0 = alpha * len(vocab)
    b0 = alpha * len(vocab)

    Na = sum(counts_a.values())
    Nb = sum(counts_b.values())

    out = []
    for w in vocab:
        ca = counts_a.get(w, 0) + alpha
        cb = counts_b.get(w, 0) + alpha
        oa = math.log(ca / (Na + a0 - ca))
        ob = math.log(cb / (Nb + b0 - cb))
        out.append((w, oa - ob))
    return out

def counter_from_tokens(series_of_tokens):
    c = Counter()
    for toks in series_of_tokens:
        c.update(toks)
    return c

cnt_aj = counter_from_tokens(df[df["outlet"] == "aljazeera"]["tokens"])
cnt_cnn = counter_from_tokens(df[df["outlet"] == "cnn"]["tokens"])

logodds = log_odds_with_prior(cnt_aj, cnt_cnn, alpha=0.01)
logodds_df = pd.DataFrame(logodds, columns=["word", "logodds_aj_minus_cnn"]).sort_values(
    "logodds_aj_minus_cnn", ascending=False
)

logodds_df.head(200).to_csv(os.path.join(OUTDIR, "log_odds_top_words_aljazeera.csv"), index=False)
logodds_df.tail(200).to_csv(os.path.join(OUTDIR, "log_odds_top_words_cnn.csv"), index=False)

# -------------------------
# WORD2VEC: compare neighbors/overlap
# -------------------------
neighbors_out = {}
overlap_out = {}

try:
    from gensim.models import Word2Vec

    aj_model = Word2Vec.load(
        "./outputs_updated/aljazeera_word2vec.model"
    )
    cnn_model = Word2Vec.load(
        "./outputs_updated/cnn_word2vec.model"
    )

    def safe_neighbors(model, w, topn=20):
        if w in model.wv:
            return [(x, float(s)) for x, s in model.wv.most_similar(w, topn=topn)]
        return []

    def neighbor_set(model, w, topn=20):
        return {x for x, _ in safe_neighbors(model, w, topn=topn)}

    def jaccard(a, b):
        if not a and not b:
            return 0.0
        return len(a & b) / len(a | b)

    W2V_TARGETS = ["gaza", "israel", "hamas", "idf", "palestine", "terrorist", "army", "netanyahu", "minister", "ministry", "defence", "president"]

    for w in W2V_TARGETS:
        neighbors_out.setdefault("aljazeera", {})[w] = safe_neighbors(aj_model, w, topn=20)
        neighbors_out.setdefault("cnn", {})[w] = safe_neighbors(cnn_model, w, topn=20)

        aj_set = neighbor_set(aj_model, w, topn=20)
        cnn_set = neighbor_set(cnn_model, w, topn=20)
        overlap_out[w] = {
            "topn": 20,
            "jaccard_overlap": jaccard(aj_set, cnn_set),
            "aljazeera_only": sorted(list(aj_set - cnn_set)),
            "cnn_only": sorted(list(cnn_set - aj_set)),
            "common": sorted(list(aj_set & cnn_set)),
        }

    with open(os.path.join(OUTDIR, "w2v_neighbors.json"), "w", encoding="utf-8") as f:
        json.dump(neighbors_out, f, ensure_ascii=False, indent=2)

    with open(os.path.join(OUTDIR, "w2v_overlap.json"), "w", encoding="utf-8") as f:
        json.dump(overlap_out, f, ensure_ascii=False, indent=2)

except Exception as e:
    with open(os.path.join(OUTDIR, "w2v_status.txt"), "w", encoding="utf-8") as f:
        f.write("Could not load or process Word2Vec models.\n")
        f.write(str(e))

print("DONE. Outputs written to:", OUTDIR)

In [30]:
# NOT VALID - JUST TEST
# -----------------------
# Paths
# -----------------------
IN_DIR = "./full_data_outputs"
FIG_DIR = "./figures"
TAB_DIR = "./tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# Files (expected from your pipeline)
SENT_SUMMARY = os.path.join(IN_DIR, "sentiment_summary_by_outlet.csv")
SENT_MONTHLY = os.path.join(IN_DIR, "sentiment_by_month_outlet.csv")  # your version has mean_ci_lo/hi
ENTITY_FOCUS = os.path.join(IN_DIR, "entity_focus.csv")
ENTITY_TESTS = os.path.join(IN_DIR, "entity_tests.csv")
LOGODDS_AJ = os.path.join(IN_DIR, "log_odds_top_words_aljazeera.csv")
LOGODDS_CNN = os.path.join(IN_DIR, "log_odds_top_words_cnn.csv")
W2V_OVERLAP = os.path.join(IN_DIR, "w2v_overlap.json")

# -----------------------
# Helpers
# -----------------------
def save_table(df: pd.DataFrame, name: str):
    out = os.path.join(TAB_DIR, name)
    df.to_csv(out, index=False)
    print("[table]", out)

def safe_read_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df

def plot_save(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    print("[fig]", path)

# -----------------------
# 1) Sentiment summary table
# -----------------------
sent_summary = safe_read_csv(SENT_SUMMARY)
# reorder columns if present
preferred = ["outlet", "n", "mean", "median", "q25", "q75", "std"]
cols = [c for c in preferred if c in sent_summary.columns] + [c for c in sent_summary.columns if c not in preferred]
sent_summary = sent_summary[cols]
save_table(sent_summary, "sentiment_summary_by_outlet_clean.csv")

# -----------------------
# 2) Sentiment over time (mean with CI)
# -----------------------
sent_month = safe_read_csv(SENT_MONTHLY)
sent_month["month"] = pd.to_datetime(sent_month["month"], errors="coerce")

# Some versions use mean_ci_lo/hi, some ci_lo/hi
lo_col = "mean_ci_lo" if "mean_ci_lo" in sent_month.columns else ("ci_lo" if "ci_lo" in sent_month.columns else None)
hi_col = "mean_ci_hi" if "mean_ci_hi" in sent_month.columns else ("ci_hi" if "ci_hi" in sent_month.columns else None)

# Plot
plt.figure(figsize=(10, 5))
for outlet, g in sent_month.sort_values("month").groupby("outlet"):
    plt.plot(g["month"], g["mean"], marker="o", label=outlet)
    if lo_col and hi_col:
        plt.fill_between(g["month"], g[lo_col], g[hi_col], alpha=0.2)

plt.axhline(0, linewidth=1)
plt.title("Sentiment over time (monthly mean VADER compound, with bootstrap CI)")
plt.xlabel("Month")
plt.ylabel("Mean compound sentiment")
plt.legend()
plot_save(os.path.join(FIG_DIR, "sentiment_by_month_with_ci.png"))

# Also export a tidy table
save_table(sent_month, "sentiment_by_month_outlet_clean.csv")

# -----------------------
# 3) Entity focus: presence % and intensity per 10k
# -----------------------
entity_focus = safe_read_csv(ENTITY_FOCUS)

# Identify entities from columns
pct_cols = [c for c in entity_focus.columns if c.startswith("pct_articles_mention_")]
rate_cols = [c for c in entity_focus.columns if c.startswith("mentions_per_10k_")]

# Make long-format tables for clean plotting / reporting
pct_long = entity_focus.melt(id_vars=["outlet"], value_vars=pct_cols, var_name="metric", value_name="pct")
pct_long["entity"] = pct_long["metric"].str.replace("pct_articles_mention_", "", regex=False)

rate_long = entity_focus.melt(id_vars=["outlet"], value_vars=rate_cols, var_name="metric", value_name="rate_per_10k")
rate_long["entity"] = rate_long["metric"].str.replace("mentions_per_10k_", "", regex=False)

save_table(pct_long.sort_values(["entity","outlet"]), "entity_presence_long.csv")
save_table(rate_long.sort_values(["entity","outlet"]), "entity_intensity_long.csv")

# Presence bar chart
entities = sorted(pct_long["entity"].unique())
x = np.arange(len(entities))
width = 0.35

aj_vals = pct_long[pct_long["outlet"]=="aljazeera"].set_index("entity").reindex(entities)["pct"].values
cnn_vals = pct_long[pct_long["outlet"]=="cnn"].set_index("entity").reindex(entities)["pct"].values

plt.figure(figsize=(12, 5))
plt.bar(x - width/2, aj_vals, width, label="aljazeera")
plt.bar(x + width/2, cnn_vals, width, label="cnn")
plt.xticks(x, entities, rotation=45, ha="right")
plt.ylabel("% of articles mentioning entity")
plt.title("Entity presence (% of articles that mention entity at least once)")
plt.legend()
plot_save(os.path.join(FIG_DIR, "entity_presence_pct.png"))

# Intensity bar chart
aj_rate = rate_long[rate_long["outlet"]=="aljazeera"].set_index("entity").reindex(entities)["rate_per_10k"].values
cnn_rate = rate_long[rate_long["outlet"]=="cnn"].set_index("entity").reindex(entities)["rate_per_10k"].values

plt.figure(figsize=(12, 5))
plt.bar(x - width/2, aj_rate, width, label="aljazeera")
plt.bar(x + width/2, cnn_rate, width, label="cnn")
plt.xticks(x, entities, rotation=45, ha="right")
plt.ylabel("Mentions per 10k tokens")
plt.title("Entity mention intensity (mentions per 10k tokens)")
plt.legend()
plot_save(os.path.join(FIG_DIR, "entity_intensity_per10k.png"))

# Entity tests table (chi-square + Cramer's V)
if os.path.exists(ENTITY_TESTS):
    entity_tests = safe_read_csv(ENTITY_TESTS)
    save_table(entity_tests.sort_values("p"), "entity_tests_sorted.csv")

# -----------------------
# 4) Log-odds: clean + plot top terms (IMPORTANT filtering)
# -----------------------
def clean_logodds(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Basic cleanup: remove tiny tokens, obvious junk like 'com'
    df["word"] = df["word"].astype(str)
    df = df[df["word"].str.len() >= 3]
    df = df[~df["word"].isin({"com", "www", "http", "https"})]
    # Remove tokens with digits
    df = df[~df["word"].str.contains(r"\d", regex=True)]
    return df

log_aj = clean_logodds(safe_read_csv(LOGODDS_AJ))
log_cnn = clean_logodds(safe_read_csv(LOGODDS_CNN))

# Save cleaned versions
save_table(log_aj, "log_odds_aljazeera_clean.csv")
save_table(log_cnn, "log_odds_cnn_clean.csv")

# Plot top 20 differentiating terms each side
top_n = 20

aj_top = log_aj.sort_values("logodds_aj_minus_cnn", ascending=False).head(top_n).iloc[::-1]
cnn_top = log_cnn.sort_values("logodds_aj_minus_cnn", ascending=True).head(top_n)  # most CNN-leaning

plt.figure(figsize=(10, 6))
plt.barh(aj_top["word"], aj_top["logodds_aj_minus_cnn"])
plt.xlabel("Log-odds (AJ minus CNN)")
plt.title(f"Top {top_n} words more characteristic of Al Jazeera (after basic cleanup)")
plot_save(os.path.join(FIG_DIR, "logodds_top_aljazeera.png"))

plt.figure(figsize=(10, 6))
plt.barh(cnn_top["word"], cnn_top["logodds_aj_minus_cnn"])
plt.xlabel("Log-odds (AJ minus CNN)")
plt.title(f"Top {top_n} words more characteristic of CNN (after basic cleanup)")
plot_save(os.path.join(FIG_DIR, "logodds_top_cnn.png"))

# -----------------------
# 5) Word2Vec overlap report (exploratory)
# -----------------------
if os.path.exists(W2V_OVERLAP):
    with open(W2V_OVERLAP, "r", encoding="utf-8") as f:
        w2v = json.load(f)

    rows = []
    for target, info in w2v.items():
        rows.append({
            "target": target,
            "topn": info.get("topn"),
            "jaccard_overlap": info.get("jaccard_overlap"),
            "n_common": len(info.get("common", [])),
            "n_aj_only": len(info.get("aljazeera_only", [])),
            "n_cnn_only": len(info.get("cnn_only", [])),
        })
    w2v_summary = pd.DataFrame(rows).sort_values("jaccard_overlap")
    save_table(w2v_summary, "w2v_overlap_summary.csv")

print("\nDONE ✅ Figures in:", FIG_DIR, "| Tables in:", TAB_DIR)

[table] ./tables/sentiment_summary_by_outlet_clean.csv
[fig] ./figures/sentiment_by_month_with_ci.png
[table] ./tables/sentiment_by_month_outlet_clean.csv
[table] ./tables/entity_presence_long.csv
[table] ./tables/entity_intensity_long.csv
[fig] ./figures/entity_presence_pct.png
[fig] ./figures/entity_intensity_per10k.png
[table] ./tables/entity_tests_sorted.csv
[table] ./tables/log_odds_aljazeera_clean.csv
[table] ./tables/log_odds_cnn_clean.csv
[fig] ./figures/logodds_top_aljazeera.png
[fig] ./figures/logodds_top_cnn.png
[table] ./tables/w2v_overlap_summary.csv

DONE ✅ Figures in: ./figures | Tables in: ./tables


In [ ]:
"""
Goal:
  Create tidy, unbiased, thesis-ready graphs + tables from your pipeline results.

Inputs (recommended):
  - articles_enriched_updated.parquet   (from your phase1 pipeline)

Optional (if present):
  - lexical_keyterms_by_month.csv
  - sentiment_by_month_outlet.csv
  - entity_focus.csv
  - w2v_neighbors.json
  - w2v_overlap.json

Outputs:
  ./figures/*.png
  ./tables/*.csv
  ./tables/*.txt

"""

# =========================
# CONFIG
# =========================
PARQUET_PATH = "articles_enriched_updated.parquet"   # change if needed

OUT_FIG = "./figures"
OUT_TAB = "./tables"
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_TAB, exist_ok=True)

FILTER_POST_OCT7 = True
OCT7 = pd.Timestamp("2023-10-07", tz="UTC")

# Entities / framing keywords
ENTITIES = ["gaza", "israel", "hamas", "idf", "palestine", "terrorist", "army", "netanyahu", "minister", "ministry", "defence", "president"]
KEY_TERMS = ["militant", "militants", "fighters", "attack", "attacks", "operation", "crisis", "settler", "settlers", "apartheid", "genocide", "terrorist", "terrorists", "ceasefire"]
# If you add more targets later, add here too:
EXTRA_TARGETS = ["terrorist", "palestinian", "israeli"]

# Tokenization / junk filtering (keep it consistent everywhere)
TOKEN_RE = re.compile(r"[a-z']+")

def load_stopwords():
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    return set(stopwords.words("english"))

STOPWORDS = load_stopwords()

# junk tokens that often come from scraping artifacts or boilerplate
JUNK = {
    "reuters", "ap", "copyright", "cookie", "cookies", "newsletter", "subscribe",
    "sign", "signs", "click", "share", "facebook", "twitter", "instagram", "youtube",
    "terms", "policy", "policies", "privacy", "advertisement", "advertising",
    "cnn", "aljazeera", "al", "jazeera",
}

def tokenize_clean(text: str):
    toks = TOKEN_RE.findall(str(text).lower())
    cleaned = []
    for t in toks:
        if len(t) < 3:
            continue
        if t in STOPWORDS:
            continue
        if t in JUNK:
            continue
        # remove tokens that are basically just apostrophes / weird remnants
        if t.strip("'") == "":
            continue
        cleaned.append(t)
    return cleaned


# =========================
# LOAD ARTICLE-LEVEL DATA
# =========================
def load_articles():
    df = pd.read_parquet(PARQUET_PATH)

    needed = {"outlet", "date", "text"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in parquet: {missing}. Found: {list(df.columns)}")

    df = df.copy()
    df["outlet"] = df["outlet"].astype(str).str.lower()
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
    df = df.dropna(subset=["date"])
    df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

    if FILTER_POST_OCT7:
        df = df[df["date"] >= OCT7].copy()

    # Sentiment column check
    if "sent_compound_clean" not in df.columns:
        raise ValueError("Expected 'sent_compound_clean' in parquet. Re-run phase1 sentiment pipeline.")

    tqdm.pandas(desc="Tokenizing (clean)")
    df["tokens"] = df["text"].progress_apply(tokenize_clean)
    df["n_tokens"] = df["tokens"].apply(len).astype(int)

    return df


# =========================
# HELPERS: effect sizes / ECDF
# =========================
def cliffs_delta(x, y):
    # P(X>Y)-P(X<Y). robust with imbalance but O(n*m).
    x = np.asarray(x); y = np.asarray(y)
    gt = 0; lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    return (gt - lt) / (len(x) * len(y))

def ecdf(a):
    a = np.asarray(a)
    a = a[~np.isnan(a)]
    x = np.sort(a)
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

def bootstrap_ci_mean(x, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    if len(x) < 2:
        return (np.nan, np.nan)
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(x, size=len(x), replace=True)
        boots.append(np.mean(samp))
    lo = np.quantile(boots, alpha/2)
    hi = np.quantile(boots, 1-alpha/2)
    return float(lo), float(hi)


# =========================
# PLOT 1: sentiment distribution
# =========================
def plot_sentiment_distribution(df):
    aj = df[df["outlet"]=="aljazeera"]["sent_compound_clean"].values
    cnn = df[df["outlet"]=="cnn"]["sent_compound_clean"].values

    # ECDF (best for imbalanced n)
    x1, y1 = ecdf(aj)
    x2, y2 = ecdf(cnn)

    plt.figure()
    plt.plot(x1, y1, label="Al Jazeera")
    plt.plot(x2, y2, label="CNN")
    plt.xlabel("VADER compound sentiment (clean text)")
    plt.ylabel("ECDF")
    plt.title("Sentiment distribution (ECDF) — post Oct 7, 2023")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "sentiment_ecdf.png"), dpi=200)
    plt.close()

    # Boxplot (still okay; just interpret with n imbalance)
    plt.figure()
    data = [
        df[df["outlet"]=="aljazeera"]["sent_compound_clean"].dropna().values,
        df[df["outlet"]=="cnn"]["sent_compound_clean"].dropna().values,
    ]
    plt.boxplot(data, labels=["Al Jazeera", "CNN"], showfliers=False)
    plt.ylabel("VADER compound sentiment (clean)")
    plt.title("Sentiment (boxplot) — post Oct 7, 2023")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "sentiment_boxplot.png"), dpi=200)
    plt.close()

    # Stats to table/text
    mw = stats.mannwhitneyu(aj, cnn, alternative="two-sided")
    cd = cliffs_delta(aj, cnn)

    with open(os.path.join(OUT_TAB, "sentiment_tests.txt"), "w", encoding="utf-8") as f:
        f.write("Sentiment tests (post Oct 7, 2023)\n")
        f.write("--------------------------------\n")
        f.write("Mann–Whitney U test (two-sided)\n")
        f.write(f"U={mw.statistic:.3f}, p={mw.pvalue:.6g}\n\n")
        f.write("Cliff's delta (AJ - CNN)\n")
        f.write(f"delta={cd:.4f}  (positive => AJ more positive than CNN)\n")


# =========================
# PLOT 2: sentiment over time (monthly mean + bootstrap CI)
# =========================
def plot_sentiment_over_time(df):
    rows = []
    for (outlet, month), g in tqdm(df.groupby(["outlet","month"]), desc="Bootstrapping monthly means"):
        m = float(np.mean(g["sent_compound_clean"]))
        lo, hi = bootstrap_ci_mean(g["sent_compound_clean"].values)
        rows.append({"outlet": outlet, "month": month, "mean": m, "ci_lo": lo, "ci_hi": hi, "n": len(g)})
    trend = pd.DataFrame(rows).sort_values(["outlet","month"])
    trend.to_csv(os.path.join(OUT_TAB, "sentiment_by_month_outlet_with_ci.csv"), index=False)

    plt.figure()
    for outlet, g in trend.groupby("outlet"):
        g = g.sort_values("month")
        x = g["month"]
        y = g["mean"]
        plt.plot(x, y, label=outlet)
        # CI band
        plt.fill_between(x, g["ci_lo"], g["ci_hi"], alpha=0.2)
    plt.axvline(pd.Timestamp("2023-10-07"), linestyle="--")
    plt.ylabel("Mean VADER compound (clean)")
    plt.xlabel("Month")
    plt.title("Monthly sentiment mean (bootstrap CI) — post Oct 7, 2023")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "sentiment_monthly_mean_ci.png"), dpi=200)
    plt.close()

    # % positive/negative per month (using your label rule: compound > 0)
    df2 = df.copy()
    df2["sent_label"] = np.where(df2["sent_compound_clean"] > 0, "pos", "neg")
    share = (
        df2.groupby(["outlet","month","sent_label"])
           .size()
           .reset_index(name="n")
    )
    totals = share.groupby(["outlet","month"])["n"].sum().reset_index(name="total")
    share = share.merge(totals, on=["outlet","month"])
    share["pct"] = 100 * share["n"] / share["total"]
    share.to_csv(os.path.join(OUT_TAB, "sentiment_label_shares_by_month.csv"), index=False)

    # Plot share (pos %)
    plt.figure()
    for outlet, g in share[share["sent_label"]=="pos"].groupby("outlet"):
        g = g.sort_values("month")
        plt.plot(g["month"], g["pct"], label=outlet)
    plt.axvline(pd.Timestamp("2023-10-07"), linestyle="--")
    plt.ylabel("% articles with positive compound")
    plt.xlabel("Month")
    plt.title("Share of positive-sentiment articles by month")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "sentiment_positive_share_by_month.png"), dpi=200)
    plt.close()


# =========================
# ENTITY FOCUS: presence % + mentions per 10k tokens
# =========================
def plot_entity_focus(df):
    # presence per article
    for ent in ENTITIES:
        df[f"has_{ent}"] = df["tokens"].apply(lambda t: 1 if ent in t else 0)

    presence = (
        df.groupby("outlet")[[f"has_{e}" for e in ENTITIES]]
          .mean()
          .reset_index()
    )
    for e in ENTITIES:
        presence[f"pct_articles_mention_{e}"] = 100 * presence[f"has_{e}"]
        presence = presence.drop(columns=[f"has_{e}"])
    presence.to_csv(os.path.join(OUT_TAB, "entity_presence_pct.csv"), index=False)

    # mention rates per 10k tokens
    def count_term(tokens, term):
        return sum(1 for x in tokens if x == term)

    for ent in ENTITIES:
        df[f"count_{ent}"] = df["tokens"].apply(lambda toks: count_term(toks, ent))

    rate_rows = []
    for outlet, g in df.groupby("outlet"):
        total_tokens = g["n_tokens"].sum()
        row = {"outlet": outlet, "total_tokens": int(total_tokens)}
        for ent in ENTITIES:
            row[f"mentions_per_10k_{ent}"] = (g[f"count_{ent}"].sum() / total_tokens) * 10000 if total_tokens else np.nan
        rate_rows.append(row)
    rates = pd.DataFrame(rate_rows)
    rates.to_csv(os.path.join(OUT_TAB, "entity_rates_per10k.csv"), index=False)

    # Plot presence
    plt.figure(figsize=(10, 4))
    x = np.arange(len(ENTITIES))
    width = 0.35
    aj = presence[presence["outlet"]=="aljazeera"].iloc[0]
    cnn = presence[presence["outlet"]=="cnn"].iloc[0]
    plt.bar(x - width/2, [aj[f"pct_articles_mention_{e}"] for e in ENTITIES], width, label="Al Jazeera")
    plt.bar(x + width/2, [cnn[f"pct_articles_mention_{e}"] for e in ENTITIES], width, label="CNN")
    plt.xticks(x, ENTITIES)
    plt.ylabel("% of articles mentioning entity")
    plt.title("Entity focus: % of articles mentioning entity (post Oct 7, 2023)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "entity_presence_pct.png"), dpi=200)
    plt.close()

    # Plot rates per 10k
    plt.figure(figsize=(10, 4))
    ajr = rates[rates["outlet"]=="aljazeera"].iloc[0]
    cnr = rates[rates["outlet"]=="cnn"].iloc[0]
    plt.bar(x - width/2, [ajr[f"mentions_per_10k_{e}"] for e in ENTITIES], width, label="Al Jazeera")
    plt.bar(x + width/2, [cnr[f"mentions_per_10k_{e}"] for e in ENTITIES], width, label="CNN")
    plt.xticks(x, ENTITIES)
    plt.ylabel("Mentions per 10k tokens")
    plt.title("Entity focus: mentions per 10k tokens (post Oct 7, 2023)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "entity_mentions_per10k.png"), dpi=200)
    plt.close()

    # Chi-square tests per entity (presence) + Cramer's V
    def cramers_v(chi2, n, r, k):
        return math.sqrt((chi2/n) / (min(r-1, k-1)))

    tests = []
    for ent in ENTITIES:
        tab = pd.crosstab(df["outlet"], df[f"has_{ent}"])
        if tab.shape[1] < 2:
            continue
        chi2, p, dof, exp = stats.chi2_contingency(tab)
        v = cramers_v(chi2, tab.to_numpy().sum(), tab.shape[0], tab.shape[1])
        tests.append({"entity": ent, "chi2": chi2, "p": p, "cramers_v": v})
    pd.DataFrame(tests).to_csv(os.path.join(OUT_TAB, "entity_presence_tests.csv"), index=False)


# =========================
# LEXICAL FRAMING: key terms per 10k tokens over time
# =========================
def plot_lexical_keyterms(df):
    terms = KEY_TERMS + EXTRA_TARGETS

    def term_count(tokens, term):
        return sum(1 for t in tokens if t == term)

    rows = []
    for (outlet, month), g in df.groupby(["outlet","month"]):
        total = g["n_tokens"].sum()
        row = {"outlet": outlet, "month": month, "n_articles": len(g), "total_tokens": int(total)}
        for term in terms:
            c = g["tokens"].apply(lambda toks: term_count(toks, term)).sum()
            row[f"{term}_per_10k"] = (c / total) * 10000 if total else np.nan
        rows.append(row)

    lex = pd.DataFrame(rows).sort_values(["outlet","month"])
    lex.to_csv(os.path.join(OUT_TAB, "lexical_keyterms_by_month.csv"), index=False)

    # Plot each term as its own figure (clean & thesis-friendly)
    for term in terms:
        plt.figure()
        for outlet, g in lex.groupby("outlet"):
            g = g.sort_values("month")
            plt.plot(g["month"], g[f"{term}_per_10k"], label=outlet)
        plt.axvline(pd.Timestamp("2023-10-07"), linestyle="--")
        plt.ylabel(f"'{term}' per 10k tokens")
        plt.xlabel("Month")
        plt.title(f"Lexical framing term over time: {term}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_FIG, f"lex_{term}_per10k_by_month.png"), dpi=200)
        plt.close()


# =========================
# LOG-ODDS: recompute robustly (fixes weird CNN list)
# =========================
def compute_log_odds(df, min_total_freq=20, min_doc_freq=5, alpha=0.01, topn=30):
    """
    Robust Monroe-style log-odds (AJ - CNN) with filters:
      - keep words with total freq >= min_total_freq across both corpora
      - keep words appearing in at least min_doc_freq documents (across both corpora)
    """
    from collections import Counter

    # doc frequency across all docs
    docfreq = Counter()
    for toks in df["tokens"]:
        docfreq.update(set(toks))

    # corpus counts by outlet
    cnt = {}
    for outlet, g in df.groupby("outlet"):
        c = Counter()
        for toks in g["tokens"]:
            c.update(toks)
        cnt[outlet] = c

    cnt_aj = cnt.get("aljazeera", Counter())
    cnt_cnn = cnt.get("cnn", Counter())

    vocab = set(cnt_aj) | set(cnt_cnn)

    # filters
    filtered = []
    for w in vocab:
        total = cnt_aj.get(w, 0) + cnt_cnn.get(w, 0)
        if total < min_total_freq:
            continue
        if docfreq.get(w, 0) < min_doc_freq:
            continue
        # keep alphabetic only (already mostly true, but safe)
        if not re.fullmatch(r"[a-z']+", w):
            continue
        filtered.append(w)

    # log-odds with informative prior
    a0 = alpha * len(filtered)
    b0 = alpha * len(filtered)
    Na = sum(cnt_aj.get(w, 0) for w in filtered)
    Nb = sum(cnt_cnn.get(w, 0) for w in filtered)

    rows = []
    for w in filtered:
        ca = cnt_aj.get(w, 0) + alpha
        cb = cnt_cnn.get(w, 0) + alpha
        oa = math.log(ca / (Na + a0 - ca))
        ob = math.log(cb / (Nb + b0 - cb))
        rows.append((w, oa - ob, cnt_aj.get(w,0), cnt_cnn.get(w,0), docfreq.get(w,0)))

    lod = pd.DataFrame(rows, columns=["word","logodds_aj_minus_cnn","count_aj","count_cnn","docfreq_all"])
    lod = lod.sort_values("logodds_aj_minus_cnn", ascending=False)

    # export full + tops
    lod.to_csv(os.path.join(OUT_TAB, "log_odds_filtered_full.csv"), index=False)
    lod.head(300).to_csv(os.path.join(OUT_TAB, "log_odds_top_words_aljazeera_filtered.csv"), index=False)
    lod.tail(300).to_csv(os.path.join(OUT_TAB, "log_odds_top_words_cnn_filtered.csv"), index=False)

    # plot topN each side
    top_aj = lod.head(topn).sort_values("logodds_aj_minus_cnn")
    top_cnn = lod.tail(topn).sort_values("logodds_aj_minus_cnn")

    plt.figure(figsize=(8, 8))
    plt.barh(top_aj["word"], top_aj["logodds_aj_minus_cnn"])
    plt.xlabel("log-odds (AJ - CNN)")
    plt.title(f"Top {topn} distinctive words: Al Jazeera (filtered)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "logodds_top_aljazeera_filtered.png"), dpi=200)
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.barh(top_cnn["word"], top_cnn["logodds_aj_minus_cnn"])
    plt.xlabel("log-odds (AJ - CNN)")
    plt.title(f"Top {topn} distinctive words: CNN (filtered)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_FIG, "logodds_top_cnn_filtered.png"), dpi=200)
    plt.close()


# =========================
# WORD2VEC: tidy export (no misleading inference)
# =========================
def export_w2v_tables():
    neighbors_path = "w2v_neighbors.json"
    overlap_path = "w2v_overlap.json"

    if not (os.path.exists(neighbors_path) and os.path.exists(overlap_path)):
        # silently skip if missing
        return

    with open(neighbors_path, "r", encoding="utf-8") as f:
        neighbors = json.load(f)
    with open(overlap_path, "r", encoding="utf-8") as f:
        overlap = json.load(f)

    # neighbors to table
    rows = []
    for outlet, words in neighbors.items():
        for target, neigh_list in words.items():
            for rank, (w, sim) in enumerate(neigh_list, start=1):
                rows.append({"outlet": outlet, "target": target, "rank": rank, "neighbor": w, "cosine": sim})
    pd.DataFrame(rows).to_csv(os.path.join(OUT_TAB, "w2v_neighbors_tidy.csv"), index=False)

    # overlap to table
    rows = []
    for target, d in overlap.items():
        rows.append({
            "target": target,
            "topn": d.get("topn"),
            "jaccard_overlap": d.get("jaccard_overlap"),
            "n_common": len(d.get("common", [])),
            "n_aj_only": len(d.get("aljazeera_only", [])),
            "n_cnn_only": len(d.get("cnn_only", [])),
        })
    pd.DataFrame(rows).to_csv(os.path.join(OUT_TAB, "w2v_overlap_summary.csv"), index=False)


# =========================
# MAIN
# =========================
def main():
    print("[1/5] Loading article-level data...")
    df = load_articles()
    print("Counts (post Oct 7):")
    print(df["outlet"].value_counts())

    print("[2/5] Plot sentiment distribution + tests...")
    plot_sentiment_distribution(df)

    print("[3/5] Plot sentiment over time...")
    plot_sentiment_over_time(df)

    print("[4/5] Plot entity focus...")
    plot_entity_focus(df)

    print("[5/5] Plot lexical framing + recompute log-odds (filtered)...")
    plot_lexical_keyterms(df)
    compute_log_odds(df, min_total_freq=20, min_doc_freq=5, alpha=0.01, topn=30)

    print("[extra] Export Word2Vec tidy tables if JSONs exist...")
    export_w2v_tables()

    print("\nDONE ✅")
    print("Figures:", OUT_FIG)
    print("Tables:", OUT_TAB)

if __name__ == "__main__":
    main()